# Week 01 · Models of Computation
## Interning, homework review, interactive Turing machines, and the halting problem

**Goals:** distinguish equality from identity; explain how a transition table executes; trace the example machines; distinguish a timeout from a proof of nontermination; explain why a universal halting decider cannot exist.

Run cells from top to bottom. Before each **Predict** prompt, discuss your answer before executing. The notebook contains worked homework solutions, so use it after submitting Week 00.

**Environment:** Python 3.11+, Jupyter/VS Code or Colab. The engine has no external dependencies. The buttons need `ipywidgets`; a plain step-by-step alternative is included. The notebook uses the repository's engine when available, otherwise an embedded snapshot of the same source and examples. No repository download is needed.


## 1. Interning: equal values, shared objects

**Predict:** if two objects compare equal, must `is` also return `True`?

`==` asks about equality; `is` asks about identity. Immutable values can sometimes share an object safely. Sharing is an implementation choice, so program correctness must not depend on incidental identity of numbers or strings.


In [ ]:
import platform
import sys
print(platform.python_implementation(), sys.version.split()[0])

for text in ['42', '1000']:
    a = int(text)  # Runtime construction, rather than two compiled literals.
    b = int(text)
    print(text, 'equal:', a == b, '; same object:', a is b)

left, right = [1, 2], [1, 2]
print('Lists:', left == right, left is right)


CPython documents a cache of integer objects from **−5 through 256**. Reusing a cached integer is related to string interning but is a separate mechanism. Identical literals can also share compiled constants; therefore observations may differ between a single cell, separate cells, and runtime construction. Do not use `is` for numeric equality. [CPython integer-object documentation](https://docs.python.org/3/c-api/long.html#c.PyLong_FromLong)

For strings, **interning** chooses a canonical object for a value. `sys.intern(s)` looks up the value in an internal table and returns the shared string. Keep the returned reference. Python commonly interns names used in programs; arbitrary equal strings need not already be interned. [Python: sys.intern](https://docs.python.org/3/library/sys.html#sys.intern)


In [ ]:
import sys
a = 'hello_world_of_python'
b = 'hello_world_of_python'
#a = '_'.join(['Hello', 'world', 'of', 'Python'])
#b = '_'.join(['Hello', 'world', 'of', 'Python'])
print('Before:', a == b, a is b)  # Observe; do not assert incidental identity.
a = sys.intern(a)
b = sys.intern(b)
print('After: ', a == b, a is b)
assert a is b


**Why do it?** Repeated strings can share storage. In dictionary lookups, interned keys and an interned lookup string can use an identity comparison after hashing. Interning itself also costs work, so it is not automatically a speedup. Use it selectively for repeatedly used names or tokens, and measure your actual workload. [Python: sys.intern](https://docs.python.org/3/library/sys.html#sys.intern)

Conceptually: `value → lookup in intern table → existing canonical string, or register one`. This is an optimization of representation, not a change to string equality.

**Discuss:** why is sharing immutable strings safe, while sharing mutable lists can change program behavior? What goes wrong with `user_input is "yes"`? Use `user_input == "yes"`; reserve identity checks for questions such as `value is None`.


In [ ]:
# Count distinct objects while keeping every reference alive.
raw = [''.join(['state', '_', str(i % 3)]) for i in range(100)]
canonical = [sys.intern(s) for s in raw]
print('Distinct values:', len(set(raw)))
print('Objects before interning:', len({id(s) for s in raw}))
print('Objects after interning:', len({id(s) for s in canonical}))
assert raw == canonical
# This is an object-count experiment, not a total-memory benchmark.


## 2. Week 00 homework: a quick review

Answers are not given here. Check the recording of the webinar for details.

## 3. A Turing machine we can inspect

A configuration consists of **state + head position + tape contents**. One transition reads a symbol, writes a symbol, moves the head, and changes state. The tape is unbounded in the mathematical model; our simulator stores only nonblank cells and has practical execution limits.

In this repository the rule format is:

```text
STATE READ -> WRITE MOVE NEXT_STATE
q0    0    -> 1     R    q0
```

`_` is blank; `L`, `R`, `S` mean left, right, stay. The head starts at position 0 in `q0`. `HALT`, `ACCEPT`, and `REJECT` are terminal states. A missing transition raises a simulator error; it is not automatically acceptance or proof of an infinite computation. Other textbook conventions may treat a missing rule as halting.

The five `.tm` files are programs.

### Setup

Run the next cell once. It imports the existing engine, without replacing its transition semantics. The embedded fallback makes this notebook usable on its own; when the repository is present, its current files take precedence.


In [ ]:
import base64
import importlib
import io
from pathlib import Path
import sys
import os
import tempfile
import zipfile

EMBEDDED_EXAMPLES = ['binary_increment.tm', '', 'copy.tm']

try:
    import turing_machine
except ImportError:
    cur_dir = os.getcwd()
    engine_root = Path(cur_dir).parent / 'tools' / 'Turing Machine'
    source_dir = engine_root / 'src'
    EXAMPLES = {name: (engine_root / 'examples' / name).read_text(encoding='utf-8')
                for name in EMBEDDED_EXAMPLES}
    engine_origin = str(engine_root)
    sys.path.insert(0, str(source_dir))
except Exception as E:
    raise E

from turing_machine import TuringMachine, parse_program
from turing_machine.errors import MissingTransitionError, StepLimitExceededError

def make_machine(name, input_data):
    machine = TuringMachine(parse_program(EXAMPLES[name]))
    machine.reset(input_data)
    return machine

print('Engine:', engine_origin)
print('Programs:', ', '.join(EXAMPLES))


### Interactive controls

The panel uses `ipywidgets`. If it is missing, uncomment and run the installation line below, then rerun the panel cells. For Colab, also uncomment the widget-manager lines if controls do not appear. A live Python kernel is required; static notebook previews cannot run buttons.


In [ ]:
# Run only if needed:
# %pip install ipywidgets
# For Google Colab, if needed:
# from google.colab import output
# output.enable_custom_widget_manager()


Choose a program, edit its input, and press **Reset / apply**. **Step** executes one rule; **Run N steps** executes a bounded batch. The highlighted rule is the *next* rule to execute. The tape window follows the head, with absolute cell indices.

Changing the selection loads that example's input and rules. Editing the input or rules takes effect only after **Reset / apply**. Every run has a 2,000-step budget; reaching it means **unknown**, not “loops forever”. Increase the budget in code only when needed.


In [ ]:
from html import escape

DEFAULT_INPUTS = {
    'invert_bits.tm': '01001',
    'binary_increment.tm': '1011',
    'unary_addition.tm': '111+11',
    'even_1.tm': '1011',
    'copy.tm': '101',
}

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

class MachinePanel:
    def __init__(self):
        self.machine = None
        self.previous = None
        self.failure = ''
        self.budget = 2000
        self.program = widgets.Dropdown(options=list(DEFAULT_INPUTS), description='Program:')
        self.input = widgets.Text(description='Input:')
        self.editor = widgets.Textarea(layout=widgets.Layout(width='100%', height='230px'))
        self.reset_button = widgets.Button(description='Reset / apply')
        self.step_button = widgets.Button(description='Step', button_style='primary')
        self.batch_button = widgets.Button(description='Run N steps')
        self.batch_size = widgets.BoundedIntText(value=20, min=1, max=500, description='N:')
        self.view = widgets.HTML()
        self.program.observe(self.select, names='value')
        self.reset_button.on_click(self.reset)
        self.step_button.on_click(lambda _: self.advance(1))
        self.batch_button.on_click(lambda _: self.advance(self.batch_size.value))
        self.controls = widgets.VBox([
            widgets.HBox([self.program, self.input]),
            widgets.HBox([self.reset_button, self.step_button, self.batch_button, self.batch_size]),
            widgets.HTML('<b>Transition editor — press Reset / apply after edits</b>'),
            self.editor, self.view,
        ])
        self.select()

    def select(self, change=None):
        name = self.program.value
        self.input.value = DEFAULT_INPUTS[name]
        self.editor.value = EXAMPLES[name]
        self.reset()

    def reset(self, button=None):
        self.machine = None
        self.previous = None
        self.failure = ''
        try:
            self.loaded_source = self.editor.value
            self.machine = TuringMachine(parse_program(self.loaded_source))
            self.machine.reset(self.input.value)
        except Exception as exc:
            self.failure = f'{type(exc).__name__}: {exc}'
        self.render()

    def advance(self, count):
        if self.machine is None or self.failure:
            return
        for _ in range(min(count, 500)):
            if self.machine.halted or self.machine.steps >= self.budget:
                break
            try:
                self.previous = self.machine.step()
            except Exception as exc:
                self.failure = f'{type(exc).__name__}: {exc}'
                break
        self.render()

    def render(self):
        m = self.machine
        disabled = (m is None or bool(self.failure) or m.halted or m.steps >= self.budget)
        self.step_button.disabled = disabled
        self.batch_button.disabled = disabled
        if m is None:
            self.view.value = '<pre>' + escape(self.failure) + '</pre>'
            return
        status = ('ERROR: ' + self.failure if self.failure else
                  m.result().status if m.halted else
                  'UNKNOWN — step budget reached' if m.steps >= self.budget else 'running')
        positions = range(m.head - 8, m.head + 9)
        indices = ''.join(f'<th style="padding:6px">{p}</th>' for p in positions)
        symbols = ''.join(
            '<td style="text-align:center;padding:8px;border:1px solid #888;'
            + ('background:#ffe082;color:#111;font-weight:bold' if p == m.head else '')
            + '">' + escape(m.tape.read(p)) + (' ↑' if p == m.head else '') + '</td>'
            for p in positions)
        next_rule = None if m.halted else m.program.transitions.get((m.state, m.tape.read(m.head)))
        rule_lines = []
        for line, text in enumerate(self.loaded_source.splitlines(), 1):
            style = 'background:#ffe082;color:#111' if next_rule and line == next_rule.line else ''
            rule_lines.append(f'<div style="{style}">{line:02d}  {escape(text)}</div>')
        previous = 'none'
        if self.previous:
            p = self.previous
            previous = f'{p.state_before} {p.read_symbol} -> {p.written_symbol} {p.move} {p.state_after}'
        next_text = ('terminal state' if m.halted else
                     f'{next_rule.state} {next_rule.read_symbol} -> {next_rule.write_symbol} {next_rule.move} {next_rule.next_state}'
                     if next_rule else 'undefined transition')
        self.view.value = (
            f'<p><b>{escape(status)}</b> · step {m.steps} / {self.budget} · state {escape(m.state)} · head {m.head}</p>'
            f'<table><tr>{indices}</tr><tr>{symbols}</tr></table>'
            f'<p>Previous: <code>{escape(previous)}</code><br>Next: <code>{escape(next_text)}</code></p>'
            f'<p>Nonblank span (outer blanks omitted): <code>{escape(repr(m.tape.normalized()))}</code></p>'
            '<pre style="max-height:300px;overflow:auto">' + ''.join(rule_lines) + '</pre>')

if widgets is not None:
    panel = MachinePanel()
    display(panel.controls)
else:
    print('ipywidgets is missing. Install it above, or use the manual controls below.')


### Manual controls / direct Python API

These cells work even without widgets. Rerun the second cell for one more step; rerun the first to reset. The brackets mark the head. Use this API when explaining what the buttons do.


In [ ]:
from turing_machine.visualize import trace_line
manual = make_machine('binary_increment.tm', '1011')
print(trace_line(manual, None))


In [ ]:
if not manual.halted and manual.steps < 2000:
    info = manual.step()
    print(trace_line(manual, info))
else:
    print('Stopped:', manual.result())


## 4. Walk through the example programs

For each example: predict the output → select it in the panel → step through the interesting transition → explain the invariant → check a boundary input.

### A. `invert_bits.tm`: one left-to-right pass

For a binary word `w`, produce its bitwise complement. Invariant: cells left of the head have been inverted; unvisited input cells are unchanged. The first blank stops the machine.

Try `01001`, `0`, and the empty input. For length `n`, how many transitions execute, including the final blank rule?


In [ ]:
print(EXAMPLES['invert_bits.tm'])
for data, expected in [('01001', '10110'), ('0', '1'), ('', '')]:
    result = make_machine('invert_bits.tm', data).run(max_steps=2000)
    assert result.output == expected and result.steps == len(data) + 1
    print(repr(data), '->', repr(result.output), 'steps:', result.steps)


### B. `binary_increment.tm`: carrying is a state

The first phase finds the right edge; `q_carry` moves left through trailing ones, replacing them with zeros. A zero absorbs the carry. If all bits were ones, a leading one is written at a negative tape index.

**Predict:** `1011 + 1`, `111 + 1`, and `0 + 1`. On empty input this particular program writes `1`; treat this as its convention, not a universal binary-number syntax rule.


In [ ]:
print(EXAMPLES['binary_increment.tm'])
for data, expected in [('1011', '1100'), ('111', '1000'), ('0', '1'), ('', '1')]:
    result = make_machine('binary_increment.tm', data).run(max_steps=2000)
    assert result.output == expected
    print(repr(data), '->', repr(result.output), 'leftmost nonblank index:', result.tape_start)


### C. `copy.tm`: mark, carry, return, restore

Output is `w@w`, not just `ww`. First write the separator `@` and return to the beginning. In `q2`, mark the current source bit as `X`; remember its value in state `r0` or `r1`; scan to the end and append that bit. Return in `l0` or `l1`, restore `X` to the original bit, and move to the next source position. Reaching `@` in `q2` means all bits were copied.

**Invariant:** at the start of each `q2` iteration, the copied suffix equals the already processed prefix. The temporary `X` is restored before the next iteration.

Trace `10` or `101` in the panel. Why are two outbound states needed? Why must the original bit be restored? The empty word produces `@`.


In [ ]:
print(EXAMPLES['copy.tm'])
for data in ['10', '101', '']:
    result = make_machine('copy.tm', data).run(max_steps=2000)
    assert result.output == data + '@' + data
    print(repr(data), '->', repr(result.output), 'steps:', result.steps)


**Short experiment:** compare copying inputs of lengths 2, 4, 8, and 16. Explain the growth using the repeated trips across the tape. Timing is unnecessary; count transitions. For this implementation, the number of full trips grows with input length and each trip crosses a span proportional to that length: quadratic growth.


In [ ]:
for n in [2, 4, 8, 16]:
    result = make_machine('copy.tm', '1' * n).run(max_steps=10000)
    print('length:', n, 'steps:', result.steps)


## 5. The halting problem

Given an arbitrary program `P` and input `x`, can one algorithm always finish and correctly decide whether `P(x)` eventually halts?

**The answer is no.** This is about a universal, always-correct, always-terminating decision procedure. It does not prevent us from proving termination or nontermination for particular programs.

### A timeout is not a verdict

A bounded simulator can answer “halted within this budget” or “unknown after this budget”. An undefined transition is a separate error under our emulator's rules.


In [ ]:
def bounded_run(source, data='', budget=100):
    machine = TuringMachine(parse_program(source))
    machine.reset(data)
    try:
        machine.run(max_steps=budget)
        return {'verdict': 'HALTED', 'steps': machine.steps, 'output': machine.tape.normalized()}
    except StepLimitExceededError:
        return {'verdict': 'UNKNOWN', 'steps': machine.steps}
    except MissingTransitionError as exc:
        return {'verdict': 'UNDEFINED TRANSITION', 'steps': machine.steps, 'detail': str(exc)}

stationary_loop = 'q0 _ -> _ S q0'
drifting_loop = 'q0 _ -> _ R q0'
for label, source, data, budget in [
    ('Increment, small budget', EXAMPLES['binary_increment.tm'], '111', 2),
    ('Increment, sufficient budget', EXAMPLES['binary_increment.tm'], '111', 100),
    ('Stationary loop', stationary_loop, '', 100),
    ('Drifting loop', drifting_loop, '', 100),
    ('Missing rule', 'q0 0 -> 0 S HALT', '', 100),
]:
    print(label, bounded_run(source, data, budget))


Both loop examples above can be proved nonterminating directly from their rules. The bounded simulator alone did not prove that. In particular, the same `UNKNOWN` label was also returned for an increment that eventually halts.

### Why a universal decider is impossible

Assume a total algorithm `H(program, input)` exists: it always finishes, returning `True` exactly when the supplied computation halts. Programs can be encoded as data, so a program description can also be used as input.

Construct `D` using this hypothetical algorithm:

```text
D(program_text):
    if H(program_text, program_text):
        loop forever
    else:
        halt
```

Now ask what happens to `D(description_of_D)`.

| H's prediction | What D does by construction | Contradiction |
| --- | --- | --- |
| It halts | Loops forever | Prediction is wrong |
| It does not halt | Halts | Prediction is wrong |

Both possibilities contradict the assumed correctness of `H`. Therefore such a total universal decider cannot exist. The pseudocode is an argument under an impossible assumption; we do not implement or execute `H` or an infinite Python loop here.

Simulation can recognize halting by eventually reaching it; on nonhalting inputs it may wait forever. This is why the halting set is recognizable but not decidable.


### Optional: can remembering configurations detect loops?

For a deterministic machine, repeating the **entire configuration** proves an infinite cycle: the next steps must repeat too. A repeated state alone is insufficient, because the head or tape may have changed.

The detector below compares exact state, head index, and every nonblank cell. It remains bounded and may answer `UNKNOWN`. It is a useful partial method, not a universal decider.


In [ ]:
def detect_repeated_configuration(source, data='', budget=200):
    machine = TuringMachine(parse_program(source))
    machine.reset(data)
    seen = set()
    for step in range(budget + 1):
        if machine.halted:
            return 'HALTED'
        configuration = (machine.state, machine.head,
                         tuple(sorted(machine.tape.snapshot().items())))
        if configuration in seen:
            return 'PROVED LOOP: repeated full configuration'
        seen.add(configuration)
        if step == budget:
            return 'UNKNOWN'
        try:
            machine.step()
        except MissingTransitionError:
            return 'UNDEFINED TRANSITION'

assert detect_repeated_configuration(stationary_loop).startswith('PROVED LOOP')
assert detect_repeated_configuration(drifting_loop) == 'UNKNOWN'
assert detect_repeated_configuration(EXAMPLES['invert_bits.tm'], '010') == 'HALTED'
print('Stationary:', detect_repeated_configuration(stationary_loop))
print('Drifting:  ', detect_repeated_configuration(drifting_loop))


The drifting machine never repeats its absolute head position, so this exact-configuration detector misses a loop that we can prove by inspection. More sophisticated methods can prove more cases, but none can decide every unrestricted program/input pair.

**Connection to finite automata:** a system with finitely many possible configurations must halt or repeat a configuration. A Turing machine has unbounded tape and head positions, so there is no universal finite bound to apply this argument. A fixed tape bound creates a different, restricted problem.

## 6. Exit questions

1. Why can `a == b` be true while `a is b` is false? What does `sys.intern` explicitly change?
2. What three components determine the next step of a deterministic Turing machine?
3. What does `even_1.tm` output for the empty input? Does it enter `ACCEPT`?
4. A machine ran for a million steps without halting. What can you conclude?
5. Why is a repeated state not sufficient to prove a loop?
6. Which assumption about `H` is contradicted by the diagonal construction?

**Try after class:** edit one rule in the panel, predict the effect, then test it. Restore the original via the program selector. Can you make the bit inverter continue moving right on blanks? Explain its behavior using the rules, not just a timeout.

## References and local materials

- [Python: sys.intern](https://docs.python.org/3/library/sys.html#sys.intern).
- [CPython integer-object cache](https://docs.python.org/3/c-api/long.html#c.PyLong_FromLong).
- [Week 00 homework](../week00_python_survival_kit/W00.tasks.ipynb).
- [Turing machine example programs](../tools/Turing%20Machine/examples/) and [engine](../tools/Turing%20Machine/src/turing_machine/machine.py).

Local links work in a full checkout. The machine examples and runtime are also embedded in this notebook for standalone use.
